# Sistema Automatizado de Consulta de Marcas e Patentes

- Projeto acadêmico em Python para automatizar consultas gratuitas de marcas e patentes, com foco em organização, rastreabilidade e facilidade de uso.

### Objetivo

- Desenvolver uma base inicial para um fluxo automatizado de consulta, permitindo evoluir depois para coleta, tratamento e exibição dos resultados de forma padronizada.

In [6]:
import os
import requests
import zipfile
import pandas as pd
import urllib3
import pandas_gbq

# Silencia os avisos de requisições HTTPS
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [7]:
# 1. CONFIGURAÇÕES GERAIS E VARIÁVEIS
DIR_RAW = "../data/raw"
DIR_OUTPUT = "../outputs"
os.makedirs(DIR_RAW, exist_ok=True)
os.makedirs(DIR_OUTPUT, exist_ok=True)

# Dicionário com os nomes dos arquivos e suas respectivas URLs diretas
URLS_INPI = {
    "MARCAS_DEPOSITANTES.csv": "https://dadosabertos.inpi.gov.br/download/marcas/MARCAS_DEPOSITANTES.csv",
    "MARCAS_DADOS_BIBLIOGRAFICOS.csv": "https://dadosabertos.inpi.gov.br/download/marcas/MARCAS_DADOS_BIBLIOGRAFICOS.csv",
    "MARCAS_CLASSIFICACOES_NICE.csv": "https://dadosabertos.inpi.gov.br/download/marcas/MARCAS_CLASSIFICACOES_NICE.csv",
    "MARCAS_CLASSIFICACOES_VIENA.csv": "https://dadosabertos.inpi.gov.br/download/marcas/MARCAS_CLASSIFICACOES_VIENA.csv"
}

# Configurações do BigQuery
PROJECT_ID = 'gen-lang-client-0182432496'
DATASET_ID = 'inpi_dados'

# Autenticação Local (mantenha o caminho exato que você ajustou e funcionou)
caminho_credencial = "../gcp_key.json" 
if os.path.exists(caminho_credencial):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = caminho_credencial
    print("✅ Credenciais do Google Cloud carregadas para teste local!")
else:
    print("⚠️ Arquivo gcp_key.json não encontrado. O envio para o BigQuery falhará.")

✅ Credenciais do Google Cloud carregadas para teste local!


In [8]:
# 2. EXTRAÇÃO (DOWNLOAD DIRETO DOS CSVS)
print(f"📥 Baixando arquivos CSV diretamente para {DIR_RAW}...")

for nome_arquivo, url in URLS_INPI.items():
    caminho_destino = os.path.join(DIR_RAW, nome_arquivo)
    print(f"   -> Baixando {nome_arquivo}...")
    
    try:
        response = requests.get(url, stream=True, verify=False)
        response.raise_for_status() # Dispara erro se a URL estiver quebrada (ex: erro 404)
        
        with open(caminho_destino, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"      Concluído!")
    except Exception as e:
        print(f"❌ Erro ao baixar {nome_arquivo}: {e}")

print("✅ Etapa de extração finalizada.")

📥 Baixando arquivos CSV diretamente para ../data/raw...
   -> Baixando MARCAS_DEPOSITANTES.csv...
      Concluído!
   -> Baixando MARCAS_DADOS_BIBLIOGRAFICOS.csv...
      Concluído!
   -> Baixando MARCAS_CLASSIFICACOES_NICE.csv...
      Concluído!
   -> Baixando MARCAS_CLASSIFICACOES_VIENA.csv...
      Concluído!
✅ Etapa de extração finalizada.


In [22]:
# 3. CARREGAMENTO DOS ARQUIVOS (LOAD)
print("📂 Carregando os arquivos CSV da pasta raw...")

csv_depositantes = os.path.join(DIR_RAW, "MARCAS_DEPOSITANTES.csv")
csv_bibliograficos = os.path.join(DIR_RAW, "MARCAS_DADOS_BIBLIOGRAFICOS.csv")
csv_nice = os.path.join(DIR_RAW, "MARCAS_CLASSIFICACOES_NICE.csv")
csv_viena = os.path.join(DIR_RAW, "MARCAS_CLASSIFICACOES_VIENA.csv")

df_depositantes_raw = pd.read_csv(csv_depositantes, sep=',', encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_biblio_raw = pd.read_csv(csv_bibliograficos, sep=',', encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_nice_raw = pd.read_csv(csv_nice, sep=',', encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_viena_raw = pd.read_csv(csv_viena, sep=',', encoding='utf-8', low_memory=False, on_bad_lines='skip')

print("✅ Todos os arquivos carregados com sucesso!")

📂 Carregando os arquivos CSV da pasta raw...
✅ Todos os arquivos carregados com sucesso!


In [23]:
# 4. TRANSFORMAÇÃO E CRUZAMENTO (TRANSFORM & MERGE)
print("⚙️ Aplicando filtros e cruzando tabelas...")

df_pr = df_depositantes_raw[(df_depositantes_raw['estado'] == 'PR')].copy()

df_pr['cnpj_limpo'] = df_pr['cnpj_cpf_titular'].apply(lambda x: ''.join(filter(str.isdigit, str(x))))

tipo_titular = df_pr['tipo_pfpj_titular'].astype('string').str.strip().str.upper()
mask_pf = tipo_titular.eq('PESSOA FÍSICA')
mask_pj = tipo_titular.eq('PESSOA JURÍDICA')
df_pr.loc[mask_pf, 'nome'] = df_pr.loc[mask_pf, 'nome'].str.title()
df_pr.loc[mask_pj, 'nome'] = df_pr.loc[mask_pj, 'nome'].str.upper()

codigos_validos = df_pr['codigo_interno'].unique()
print(f"   -> Encontrados {len(codigos_validos)} códigos internos únicos no PR.")

df_biblio_filtrado = df_biblio_raw[df_biblio_raw['codigo_interno'].isin(codigos_validos)].copy()
df_nice_filtrado = df_nice_raw[df_nice_raw['codigo_interno'].isin(codigos_validos)].copy()
df_viena_filtrado = df_viena_raw[df_viena_raw['codigo_interno'].isin(codigos_validos)].copy()

del df_depositantes_raw, df_biblio_raw, df_nice_raw, df_viena_raw

dict_agrupamento_nice = {col: lambda x: ' | '.join(x.dropna().astype(str).unique()) for col in df_nice_filtrado.columns if col != 'codigo_interno'}
df_nice_agrupado = df_nice_filtrado.groupby('codigo_interno').agg(dict_agrupamento_nice).reset_index()

df_final = pd.merge(df_pr, df_biblio_filtrado, on='codigo_interno', how='left')
df_final = pd.merge(df_final, df_nice_agrupado, on='codigo_interno', how='left')

⚙️ Aplicando filtros e cruzando tabelas...


   -> Encontrados 433742 códigos internos únicos no PR.


In [24]:
# 5. LIMPEZA E ENGENHARIA DE REGRAS DE NEGÓCIO
print("🧹 Criando regras de negócio...")

if 'numero_inpi_x' in df_final.columns:
    df_final.rename(columns={'numero_inpi_x': 'numero_processo_inpi'}, inplace=True)

colunas_para_remover = ['numero_inpi_y', 'numero_inpi', 'edicao_nice', 'especificacao_trad']
colunas_presentes = [col for col in colunas_para_remover if col in df_final.columns]
if colunas_presentes:
    df_final.drop(columns=colunas_presentes, inplace=True)

colunas_de_data = ['data_deposito', 'data_publicacao', 'data_concessao', 'data_vigencia']
for col in colunas_de_data:
    if col in df_final.columns:
        # Removido o strftime. Agora o Pandas mantém o tipo nativo "datetime64"
        df_final[col] = pd.to_datetime(df_final[col], errors='coerce')

def classificar_status(status):
    status_str = str(status).lower()
    if 'vigor' in status_str: return '🟢 Ativo'
    elif 'arquivado' in status_str or 'extinto' in status_str or 'indeferido' in status_str: return '🔴 Atenção / Recuperação'
    elif 'oposição' in status_str: return '🟡 Risco (Oposição)'
    else: return '🔵 Em Andamento Normal'

if 'descricao_situacao' in df_final.columns:
    df_final['macro_status'] = df_final['descricao_situacao'].apply(classificar_status)

hoje = pd.Timestamp.now()
def classificar_vencimento(data_vig):
    if pd.isna(data_vig): return '⚪ Não Aplicável'
    try:
        data_vig_dt = pd.to_datetime(data_vig)
        dias_restantes = (data_vig_dt - hoje).days
        if dias_restantes < -180: return '⚫ Extinto (Perda Definitiva)'
        elif -180 <= dias_restantes < 0: return '🔴 Prazo Extraordinário (Vencido, sujeito a multa)'
        elif 0 <= dias_restantes <= 180: return '🟠 Urgência (Expira em menos de 6 meses)'
        elif 180 < dias_restantes <= 365: return '🟡 Janela Aberta (Expira em 6 a 12 meses)'
        else: return '🟢 Vigente (Mais de 1 ano)'
    except:
        return '⚪ Erro na Data'

if 'data_vigencia' in df_final.columns:
    df_final['alerta_vencimento'] = df_final['data_vigencia'].apply(classificar_vencimento)

tipo_titular_final = df_final['tipo_pfpj_titular'].astype('string').str.strip().str.upper()

df_juridica = df_final[tipo_titular_final == 'PESSOA JURÍDICA'].copy()
df_nao_juridica = df_final[tipo_titular_final == 'PESSOA FÍSICA'].copy()

print("✅ Base higienizada e separada em PJ e PF com sucesso.")

🧹 Criando regras de negócio...
✅ Base higienizada e separada em PJ e PF com sucesso.


In [25]:
contagem_pf = mask_pf.sum()
contagem_pj = mask_pj.sum()
print(f"Quantidade de pessoas físicas: {contagem_pf}")
print(f"Quantidade de pessoas jurídicas: {contagem_pj}")

Quantidade de pessoas físicas: 72368
Quantidade de pessoas jurídicas: 363400


In [26]:
# 6. CARGA PARA O BIGQUERY
print("🚀 Enviando dados para o Google BigQuery...")

# Envia Pessoas Jurídicas
pandas_gbq.to_gbq(df_juridica, f'{DATASET_ID}.fat_marcas_pr_pj', project_id=PROJECT_ID, if_exists='replace')
print("-> 🏢 Base PJ salva com sucesso no BigQuery!")

# Envia Pessoas Físicas
pandas_gbq.to_gbq(df_nao_juridica, f'{DATASET_ID}.fat_marcas_pr_pf', project_id=PROJECT_ID, if_exists='replace')
print("-> 👤 Base PF salva com sucesso no BigQuery!")

# Envia Dimensão Viena
pandas_gbq.to_gbq(df_viena_filtrado, f'{DATASET_ID}.dim_viena_pr', project_id=PROJECT_ID, if_exists='replace')
print("-> 🎨 Dimensão Viena salva com sucesso no BigQuery!")

🚀 Enviando dados para o Google BigQuery...
-> 🏢 Base PJ salva com sucesso no BigQuery!
-> 👤 Base PF salva com sucesso no BigQuery!
-> 🎨 Dimensão Viena salva com sucesso no BigQuery!
